In [1]:
# Model test: Qwen2.5-7B-Instruct (originally tried Mistral-7B, unavailable via free API)
# Date: 2026-07-13
# Testing via HuggingFace Inference API

In [2]:
# -----------------------------------------------------------
# CELL 1: Load API Key and Connect to Database
# os, dotenv  → load the API key securely from .env file
# sqlite3     → connect to the Chinook database
# openai      → official Python client for the OpenAI API
# -----------------------------------------------------------

import os
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI

# Load API key from .env file
load_dotenv()
client = OpenAI()

# Connect to Chinook database
conn = sqlite3.connect('../data/Chinook_Sqlite.sqlite')
cursor = conn.cursor()

print('API key loaded and database connected successfully!')

API key loaded and database connected successfully!


In [3]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()
hf_client = InferenceClient(token=os.getenv("HUGGINGFACE_API_KEY"))

print("HF client ready!")

HF client ready!


In [4]:
# -----------------------------------------------------------
# CELL 2: Load Schema String
# We re-run the schema extraction logic here so this notebook
# is self-contained and does not depend on running another
# notebook first. The schema_string is passed into the LLM
# prompt so the model knows what tables and columns exist.
# -----------------------------------------------------------

# Extract all table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = [row[0] for row in cursor.fetchall()]

# Extract columns for each table
schema = {}
for table in tables:
    cursor.execute(f'PRAGMA table_info({table})')
    columns = cursor.fetchall()
    schema[table] = [(col[1], col[2]) for col in columns]

# Format schema as a clean string for the LLM prompt
def format_schema_for_prompt(schema):
    schema_str = ''
    for table, columns in schema.items():
        schema_str += f'Table: {table}\n'
        for col_name, col_type in columns:
            schema_str += f'  - {col_name} ({col_type})\n'
        schema_str += '\n'
    return schema_str

schema_string = format_schema_for_prompt(schema)
print('Schema loaded successfully!')
print(f'Total tables: {len(schema)}')


Schema loaded successfully!
Total tables: 11


In [5]:
# -----------------------------------------------------------
# CELL 3: Build the Prompt
# The prompt is the instruction we send to the LLM.
# It has 3 parts:
#   1. The database schema (what tables/columns exist)
#   2. Instructions (return only SQL, no explanation)
#   3. The user's question
# This is the core of prompt engineering for our project.
# -----------------------------------------------------------

def build_prompt(schema_string, user_question):
    prompt = f"""You are an expert SQL assistant.

You are given the following database schema:
{schema_string}

Write a valid SQLite SQL query to answer the following question:
{user_question}

Rules:
- Return ONLY the SQL query, nothing else
- Do not include any explanation or markdown
- Do not use ```sql or ``` in your response
- Make sure the query is valid SQLite syntax
"""
    return prompt

# Test the prompt with a sample question
test_question = "Which artist has the most albums?"
prompt = build_prompt(schema_string, test_question)
print('Prompt built successfully!')
print('---')
print(prompt)


Prompt built successfully!
---
You are an expert SQL assistant.

You are given the following database schema:
Table: Album
  - AlbumId (INTEGER)
  - Title (NVARCHAR(160))
  - ArtistId (INTEGER)

Table: Artist
  - ArtistId (INTEGER)
  - Name (NVARCHAR(120))

Table: Customer
  - CustomerId (INTEGER)
  - FirstName (NVARCHAR(40))
  - LastName (NVARCHAR(20))
  - Company (NVARCHAR(80))
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Email (NVARCHAR(60))
  - SupportRepId (INTEGER)

Table: Employee
  - EmployeeId (INTEGER)
  - LastName (NVARCHAR(20))
  - FirstName (NVARCHAR(20))
  - Title (NVARCHAR(30))
  - ReportsTo (INTEGER)
  - BirthDate (DATETIME)
  - HireDate (DATETIME)
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Em

In [6]:
test_question = "Which artist has the most albums?"
prompt = build_prompt(schema_string, test_question)

response = hf_client.chat_completion(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200
)

qwen_sql = response.choices[0].message.content.strip()
print(qwen_sql)

SELECT Artist.Name
FROM Artist
JOIN Album ON Artist.ArtistId = Album.ArtistId
GROUP BY Artist.Name
ORDER BY COUNT(Album.AlbumId) DESC
LIMIT 1


In [7]:
# -----------------------------------------------------------
# CELL 4: Send Prompt to LLM
# We send the prompt to GPT-4o-mini via the OpenAI API.
# The model reads the schema + question and returns a SQL query.
# We extract just the text content from the response.
# -----------------------------------------------------------

def get_sql_from_llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'You are an expert SQL assistant. Return only valid SQL queries.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0  # temperature=0 means deterministic output (less creative, more precise)
    )
    # Extract the SQL text from the response
    sql_query = response.choices[0].message.content.strip()
    return sql_query

# Test with our sample question
sql_query = get_sql_from_llm(prompt)
print('SQL generated by LLM:')
print(sql_query)


SQL generated by LLM:
SELECT Artist.Name, COUNT(Album.AlbumId) AS AlbumCount
FROM Artist
JOIN Album ON Artist.ArtistId = Album.ArtistId
GROUP BY Artist.ArtistId
ORDER BY AlbumCount DESC
LIMIT 1;


In [8]:
# -----------------------------------------------------------
# CELL 5: Run SQL on the Database
# We execute the SQL query the LLM generated on the Chinook
# database and fetch the results.
# If the SQL is invalid, this will raise an error — we will
# handle that in the self-correction loop later.
# -----------------------------------------------------------

def run_sql(sql_query):
    cursor.execute(sql_query)
    results = cursor.fetchall()
    return results

# Original pipeline included a demo call here, omitted for this notebook


In [9]:
qwen_results = run_sql(qwen_sql)
print(qwen_results)

[('Iron Maiden',)]


In [10]:
test_question_2 = "What are the top 5 genres by number of tracks?"
prompt_2 = build_prompt(schema_string, test_question_2)

response_2 = hf_client.chat_completion(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[{"role": "user", "content": prompt_2}],
    max_tokens=200
)

qwen_sql_2 = response_2.choices[0].message.content.strip()
print(qwen_sql_2)

SELECT Genre.Name, COUNT(*) AS TrackCount
FROM Genre
JOIN Track ON Genre.GenreId = Track.GenreId
GROUP BY Genre.Name
ORDER BY TrackCount DESC
LIMIT 5


In [11]:
qwen_results_2 = run_sql(qwen_sql_2)
print(qwen_results_2)

[('Rock', 1297), ('Latin', 579), ('Metal', 374), ('Alternative & Punk', 332), ('Jazz', 130)]


In [12]:
test_question_3 = "How many customers are from the USA?"
prompt_3 = build_prompt(schema_string, test_question_3)

response_3 = hf_client.chat_completion(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[{"role": "user", "content": prompt_3}],
    max_tokens=200
)

qwen_sql_3 = response_3.choices[0].message.content.strip()
print(qwen_sql_3)

SELECT COUNT(*) FROM Customer WHERE Country = 'USA'


In [13]:
qwen_results_3 = run_sql(qwen_sql_3)
print(qwen_results_3)

[(13,)]


In [14]:
# Note: original core_pipeline.ipynb includes cells for the full pipeline function,
# self-correction loop, and error-handling tests (Cells 6-10).
# Omitted here as out of scope for this model comparison notebook.